# 横磁場イジングモデル

## 横磁場イジングモデルとは

ある一つの方向のイジング相互作用に対して、それと垂直な向きの磁場を組み合わせたもの。

今回は以下のモデルを考える。

\begin{equation}
H = -J\sum_i{Z_i Z_{i+1}} -h\sum_i{X_i}
\end{equation}

- **第1項について**

    この項は隣り合うスピンを揃えようとする相互作用を表しています。

    実際に計算すると
    
    \begin{equation}
    \begin{aligned}
    Z_1 Z_0\ket{00} &= +1\ket{00} \\
    Z_1 Z_0\ket{11} &= +1\ket{11} \\
    Z_1 Z_0\ket{01} &= -1\ket{00} \\
    Z_1 Z_0\ket{10} &= -1\ket{11} \\
    \end{aligned}
    \end{equation}

    これから隣同士のスピンが揃っているとエネルギーが小さくなり、揃っていないとエネルギーが大きくなる。すなわちスピンが揃っている方が好まれるモデルになっている。

- **第2項について**

    スピンをZの軸からXの軸へ傾けるような量子揺らぎを表しています。


## 今回のゴール

$H$の基底エネルギー$E_0$とクエンチ後の横磁化$m_x(t)$の時間発展を求める。

## 今回行う解き方
1.  **厳密対角化**

    $2^n\times2^n$行列を対角化する。

1.  **Jordan-Wigner変換を用いてフェルミオンの問題に置き換える方法**

    フェルミオンの問題に置き換えて考えることで厳密に解くことができる。
    大まかな流れは

1.  **Trotter分解を用いて量子回路で解く方法**

    量子回路を用いて解くことができるが、誤差の逆数のオーダーのゲート数が必要。

## 1.厳密対角化の一般論

### 1-1 基底エネルギー$E_0$

$H$を対角化し、その固有値の最小値が基底エネルギー$E_0$である。

### 1-2 横磁化$m_x(t)$

$m_x(t)$は以下のように計算できる。

\begin{equation}
m_x(t) = \bra{\psi(t)}M_x\ket{\psi(t)}, M_x = \sum_{j=1}^{n}{X_j}
\end{equation}

$X_j$はそれぞれのサイトのx成分を測る演算子であり、jサイト目が
- $\ket{+}$のとき$+1$
- $\ket{-}$のとき$-1$

になるもの。$M_x$はそれの平均であると考えるとわかりやすい。

以下で$\psi(t)$を求める。シュレンディンガー方程式より

\begin{equation}
i\frac{d}{dt}\ket{\psi(t)} = H\ket{\psi(t)} \Rightarrow \ket{\psi(t)} = e^{-iHt}\ket{\psi_9}
\end{equation}

ここで初期状態$\psi_0$を固有基底で展開する。

\begin{equation}
\psi_0 = \sum_n c_n\ket{n}, c_n = \bra{n}\ket{\psi_0}
\end{equation}

これを先ほどの式に代入すると

\begin{equation}
\begin{aligned}
\ket{\psi(t)} 
&= e^{-iHt}\ket{\psi_9} \\
&= \sum_n e^{-iHt}c_n\ket{n} \\
&= \sum_n e^{-iE_{n}t}c_n\ket{n} (\because H\ket{n}=E_n\ket{n})\\
\end{aligned}
\end{equation}

このように固有状態の時間発展が追いやすいことを用いる。

これらより横磁化$m_x(t)$は

\begin{equation}
\begin{aligned}
m_x(t)
&= \bra{\psi(t)}M_x\ket{\psi(t)} \\
&= \sum_{m, n} c_m^{*}c_n e^{i(E_m - E_n)t}\bra{m}M_x\ket{n}
\end{aligned}
\end{equation}

である。

### 1-3 コードで確かめる

In [ ]:
import numpy as np
from functools import reduce

# 基本的な演算子
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
def site_op(op, i, n):
    """i番目のサイトだけにopが作用する2^n次元の行列。すなわち、I⊗I⊗...⊗op⊗...⊗Iを返す。"""
    if i < 0 or i >= n:
        raise ValueError("i must be in the range [0, n-1]")
    op_list = [I] * n
    op_list[i] = op
    return reduce(np.kron, op_list)

# ハミルトニアン
def hamiltonian(n, J, h):
    """1次元のイジングモデルのハミルトニアンを返す。"""
    dim = 2**n
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(n):
        H -= J * site_op(Z, i, n) @ site_op(Z, (i + 1) % n, n)  # 隣接するスピン間の相互作用
        H -= h * site_op(X, i, n)  # 外部磁場
    return H

[[0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
 [0.+0.j 0.+0.j 0.+0.j 1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]]


## 2. Jordan-Wigner変換を用いてフェルミオンの問題に置き換える方法

## 3. Trotter分解を用いて量子回路で解く方法